**Importing Library**

In [ ]:
import torch
from transformers import AutoTokenizer,AutoModelForCausalLM

**Loading Model**

In [ ]:
model_name="gpt2"
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

**Generating Loop**

In [ ]:
def generate(prompt,max_token=50,temperature=0.8,greedy=False):
  input_ids=tokenizer(prompt,return_tensors="pt").input_ids
  for _ in range(max_token):
    with torch.no_grad():
      outputs=model(input_ids)
      logits=outputs.logits

    last_token_logits=logits[:,-1,:]

    if greedy:
      next_token_id=torch.argmax(last_token_logits,dim=-1,keepdim=True)
    else:
      scaled_logits=last_token_logits/temperature
      probs=torch.softmax(scaled_logits,dim=-1)
      next_token_id = torch.multinomial(probs, num_samples=1)
    input_ids = torch.cat([input_ids, next_token_id], dim=1)
  return tokenizer.decode(input_ids[0], skip_special_tokens=True)

**Testing**

In [ ]:
if __name__ == "__main__":
    prompt = input("Enter prompt: ")
    output = generate(prompt,max_token=50,temperature=0.8,greedy=True)
    print("\nGenerated Text:\n")
    print(output)